<a href="https://colab.research.google.com/github/mahidodiya/evolve/blob/main/LLaVA_NeXT_Video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 30.7 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import BitsAndBytesConfig, LlavaNextVideoForConditionalGeneration, LlavaNextVideoProcessor

model_id = "llava-hf/LLaVA-NeXT-Video-7B-hf"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

processor = LlavaNextVideoProcessor.from_pretrained(model_id)
model = LlavaNextVideoForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb,
    device_map="auto",
)
print("Model loaded")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


processor_config.json:   0%|          | 0.00/209 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/741 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/687 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Model loaded


In [3]:
import av, gc
import numpy as np

PROMPT = "Describe what is happening in this video in detail."  # one prompt for all videos
NUM_FRAMES = 8   # 8 is safe on T4; try 12-16 if memory allows

def read_frames(path, num_frames=NUM_FRAMES):
    container = av.open(path)
    stream = container.streams.video[0]
    total = stream.frames
    if not total:  # some files don't report frame count
        total = sum(1 for _ in container.decode(video=0))
        container.seek(0)
    indices = set(np.linspace(0, total - 1, num_frames).astype(int).tolist())
    frames = []
    container.seek(0)
    for i, frame in enumerate(container.decode(video=0)):
        if i in indices:
            frames.append(frame.to_ndarray(format="rgb24"))
    container.close()
    return np.stack(frames)

def describe_video(path, max_new_tokens=250):
    clip = read_frames(path)

    conversation = [{
        "role": "user",
        "content": [
            {"type": "text", "text": PROMPT},
            {"type": "video"},
        ],
    }]
    prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)

    inputs = processor(text=prompt, videos=clip, padding=True, return_tensors="pt").to(model.device)
    print("Input tokens:", inputs["input_ids"].shape[-1])

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    text = processor.decode(output[0], skip_special_tokens=True)
    del inputs, output
    gc.collect(); torch.cuda.empty_cache()
    return text.split("ASSISTANT:")[-1].strip()

In [4]:
print(describe_video("movie_clip1.mp4"))

Input tokens: 1174
In the video, we see a scene from a TV show where two women are engaged in a conversation. One woman is seated on a couch, wearing a black blazer and a necklace, while the other woman is standing next to her, dressed in a blue blazer and a necklace. The standing woman is holding a phone to her ear, suggesting she is in the middle of a phone call. The seated woman is looking at the standing woman with a concerned expression, possibly indicating that she is listening to the phone call or waiting for the call to end. The setting appears to be a formal or elegant room, with a fireplace in the background and a chandelier hanging from the ceiling. The room is well-lit, and there are books and a vase on a table nearby. The atmosphere seems to be serious and possibly tense, as the standing woman's expression is focused and the seated woman's is concerned. The scene is a typical representation of a dramatic TV show moment, with the characters' body language and expressions co

In [5]:
print(describe_video("movie_hindi.mp4"))

Input tokens: 1174
In the video, we see a group of people gathered in a room, with one woman standing in the center. She is wearing a red dress and has a smile on her face. The woman is holding a man's head in her hands, and he is lying on his back on a table. The man is wearing a white shirt and appears to be asleep. The woman is gently shaking the man's head, and he is smiling and appears to be in a deep sleep. The room is dimly lit, and there are other people in the background, but they are not the focus of the scene. The woman's actions suggest that she is trying to wake the man up, but he seems to be in a deep sleep and is not responding. The atmosphere is calm and peaceful, with no signs of urgency or distress. The woman's smile and the man's relaxed posture indicate a sense of care and affection between them. The scene is a quiet moment captured in the video, with no other significant actions or events happening.


In [6]:
print(describe_video("cube.mp4"))

Input tokens: 1174
In the video, a person is seen holding a Rubik's Cube puzzle in their hands. The person is in the process of solving the puzzle, and they are in the middle of a move, with their fingers positioned in such a way that they are about to twist one of the cube's sides. The cube is a standard 3x3 Rubik's Cube, which is a popular puzzle consisting of six faces, each with nine colored squares, and it is solved by twisting the cube's sides to align all of the colored squares on each face. The person's fingers are positioned in a way that suggests they are about to twist one of the cube's sides, which is a common move in solving the puzzle. The Rubik's Cube is a popular puzzle that requires a lot of problem-solving skills and is often used as a brain teaser or a party game. The person's fingers are positioned in a way that suggests they are about to twist one of the cube's sides, which is a common move in solving the puzzle. The Rubik's Cube is a popular puzzle that requires
